<small>Dataset: Student_Performance.csv</small>

# Q13. Regression With and Without PCA

Measure how PCA changes regression accuracy, feature count and fitting time.


## Question from Classroom

Compare Regression Models With and Without PCA 
Problem statement: Investigate the effect of dimensionality reduction on the performance 
of a regression model. 
Tasks to be performed: 
1. Select a numerical regression dataset containing multiple input features. 
2. Preprocess the data and create training and testing sets. 
3. Train a regression model using the original features. 
4. Apply standardization and PCA using only the training data to avoid data leakage. 
5. Train the same regression model on the PCA-transformed features. 
6. Evaluate both models using R², RMSE and MAE on the test data. 
7. Compare predictive performance, number of features and computational 
implications.


## Understand before coding

Use four numerical student predictors -> hold out test rows -> standardize -> train original and PCA regressions -> compare metrics.


Run the code cells from top to bottom with **Shift + Enter**. If an import is missing, run the following installation line once in a new cell, then restart the kernel. The line below is a comment until you remove `#`.


**Optional package setup (run once in a new Jupyter cell if needed):**

```python
%pip install pandas numpy matplotlib scikit-learn openpyxl
```


**Code layout:** 7 listed tasks → 7 code blocks. Run them in order. Imports and helper definitions are included in the relevant task. Where the PDF repeats a task number, these blocks follow its listed order. Learned preprocessing is fitted only after splitting.


## Task 1. Select a numerical regression dataset containing multiple input features.

This block implements the task above. Variables from earlier blocks are reused.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error

df=pd.read_csv("data/Student_Performance.csv")
features=["Hours Studied","Previous Scores","Sleep Hours","Sample Question Papers Practiced"]
X=df[features]
y=df["Performance Index"]
print("Numeric predictors:",features)


Numeric predictors: ['Hours Studied', 'Previous Scores', 'Sleep Hours', 'Sample Question Papers Practiced']


## Task 2. Preprocess the data and create training and testing sets.

This block implements the task above. Variables from earlier blocks are reused.


In [2]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
medians=X_train.median()
X_train=X_train.fillna(medians)
X_test=X_test.fillna(medians)


## Task 3. Train a regression model using the original features.

This block implements the task above. Variables from earlier blocks are reused.


In [3]:
from time import perf_counter
started=perf_counter()
original_model=LinearRegression().fit(X_train,y_train)
original_seconds=perf_counter()-started


## Task 4. Apply standardization and PCA using only the training data to avoid data leakage.

This block implements the task above. Variables from earlier blocks are reused.


In [4]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
train_scaled=scaler.fit_transform(X_train)
test_scaled=scaler.transform(X_test)

from sklearn.decomposition import PCA
pca=PCA(n_components=0.95)
started=perf_counter()
train_pca=pca.fit_transform(train_scaled)
test_pca=pca.transform(test_scaled)
print("Explained variance:",pca.explained_variance_ratio_)


Explained variance: [0.25404353 0.2525403  0.24956289 0.24385328]


## Task 5. Train the same regression model on the PCA-transformed features.

This block implements the task above. Variables from earlier blocks are reused.


In [5]:
pca_model=LinearRegression().fit(train_pca,y_train)
pca_seconds=perf_counter()-started


## Task 6. Evaluate both models using R², RMSE and MAE on the test data.

This block implements the task above. Variables from earlier blocks are reused.


In [6]:
results=[]
for name,model,features_test in [("Without PCA",original_model,X_test),("With PCA",pca_model,test_pca)]:
    pred=model.predict(features_test)
    results.append({"Model":name,"R2":r2_score(y_test,pred),
      "RMSE":np.sqrt(mean_squared_error(y_test,pred)),"MAE":mean_absolute_error(y_test,pred)})
print(pd.DataFrame(results).to_string(index=False))


      Model       R2     RMSE      MAE
Without PCA 0.988714 2.045056 1.629673
   With PCA 0.988714 2.045056 1.629673


## Task 7. Compare predictive performance, number of features and computational implications.

This block implements the task above. Variables from earlier blocks are reused.


In [7]:
print("Original feature count:",X.shape[1],"PCA component count:",pca.n_components_)
print("Original fit seconds:",original_seconds,"PCA+fit seconds:",pca_seconds)
print("Discuss error vs feature reduction; one timing run does not prove a speed advantage.")


Original feature count: 4 PCA component count: 4
Original fit seconds: 0.0017742000018188264 PCA+fit seconds: 0.002013600002101157
Discuss error vs feature reduction; one timing run does not prove a speed advantage.


## Viva preparation guide



**1. What are X and y?**

X contains the predictor columns; y contains the target we want to predict. X is a 2D table and y is a 1D series.

**2. Why split training and testing data?**

Fit using the training examples. Evaluate on separate examples to estimate how well the model generalizes.

**3. What do fit() and predict() mean?**

fit() learns parameters from training examples. predict() applies the learned model to new feature rows.

**4. Why random_state=42?**

It makes random splits or model choices reproducible. The number 42 has no special mathematical meaning.

**5. What is data leakage?**

Information unavailable for a real prediction accidentally influences training. Examples: including the answer in X or fitting a scaler on test data. Fit learned preprocessing only on training data.

**6. What is overfitting?**

The model learns training-specific details/noise and does worse on unseen data. Control complexity and use validation or cross-validation.

**7. What is a hyperparameter?**

A setting chosen outside model fitting, such as max_depth, K or the number of trees. Choose it using training validation, not by repeatedly optimizing the final test result.

**8. How do I fix FileNotFoundError or KeyError?**

Open Jupyter in the praceice folder, keep data/ beside this notebook, and print df.columns.tolist() to check exact column names. Do not change X and y row counts separately.

**9. What is linear regression?**

It fits y_hat = b0 + b1*x1 + ... + bp*xp, normally by minimizing the sum of squared training residuals.

**10. What are intercept and coefficients?**

The intercept is the predicted value when encoded inputs are zero. A coefficient describes the change in prediction per unit of that feature, holding the others fixed; association does not prove causation.

**11. Define MAE and RMSE.**

MAE=mean(abs(y-y_hat)). RMSE=sqrt(mean((y-y_hat)^2)). Both use the target units; lower is better. RMSE penalizes large errors more strongly.

**12. What is R2?**

R2=1-SSE/SST. A perfect prediction scores 1. A negative score means worse squared error than predicting the test target mean. It is not accuracy or a percentage of correct records.

**13. How do I interpret actual vs predicted?**

Points near the diagonal indicate small errors. Systematic curves or large scatter suggest model limitations. Do not invent a good score when the dataset has weak signal.

**14. What does PCA do?**

Principal Component Analysis replaces correlated input features by orthogonal components ordered by explained variance. It does not use the target.

**15. Why standardize before PCA?**

Variance depends on units. Scaling prevents a large-unit feature from dominating only because of its measurement scale.

**16. What does n_components=0.95 mean?**

Retain enough principal components to explain at least about 95% of training feature variance. This does not mean 95% prediction accuracy.

**17. Where is PCA fitted?**

On standardized training features only. Apply that fitted PCA to the test features using transform().

**18. What makes this comparison fair?**

Both models use the same split, preprocessing statistics, regressor type and test metrics. Only PCA is added to the second model.

**19. Does PCA always improve prediction?**

No. A low-variance direction may be predictive of y, so discarding it can hurt performance. Compare R2, RMSE and MAE rather than assuming an improvement.

**20. Does fewer components guarantee a faster complete workflow?**

No. PCA itself costs time. The table reports a single timing including PCA fitting/transformation for the PCA branch; timing also depends on hardware and warm-up.

**21. Is PCA feature selection?**

No. Each component is a linear combination of original features, so interpretation is less direct.


### Task-to-code map

- **Block 1:** Select a numerical regression dataset containing multiple input features.
- **Block 2:** Preprocess the data and create training and testing sets.
- **Block 3:** Train a regression model using the original features.
- **Block 4:** Apply standardization and PCA using only the training data to avoid data leakage.
- **Block 5:** Train the same regression model on the PCA-transformed features.
- **Block 6:** Evaluate both models using R², RMSE and MAE on the test data.
- **Block 7:** Compare predictive performance, number of features and computational implications.
